# Entrenar un MLP en Keras

**Sesión 2 · IA Intermedio · EstudIA 2026**

Vamos a entrenar una red neuronal desde cero para un problema de clasificación con datos en tabla.

**Los datos.** 4.424 estudiantes de pregrado de una institución de educación superior en Portugal
(Realinho y otros, 2021). De cada estudiante conocemos su situación al ingresar (programa, notas
previas, edad, si tiene beca o deudas, estudios y ocupación de los padres), su desempeño en el primer
y el segundo semestre, y tres indicadores económicos del país. Con eso queremos predecir cómo termina
cuando se cumple la duración normal de su carrera:

| Clase | Qué significa |
|---|---|
| `abandono` | Dejó la carrera |
| `matriculado` | Sigue estudiando, pero no se graduó a tiempo |
| `graduado` | Se graduó a tiempo |

**El recorrido.**

| Parte | Qué hacemos |
|---|---|
| 1 | Preparar los datos y partirlos en entrenamiento, validación y prueba |
| 2 | Entrenar un primer MLP y leer sus curvas |
| 3 | Ajustar la red y volver a entrenar |
| 4 | Entrenar el modelo final y evaluarlo una sola vez con la prueba |

**Antes de empezar.** *Archivo › Guardar una copia en Drive*. No necesitas GPU. Ejecuta la celda de
preparación (▶ o `Shift + Enter`).

In [ ]:
# Preparación: librerías, datos y semillas. Ejecútala una vez.
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
ARCHIVOS = os.environ.get("ESTUDIA_ARCHIVOS", "https://storage.googleapis.com/estud-ia-course-estudia-lab/s02")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 40)
datos = pd.read_csv(f"{ARCHIVOS}/estudiantes.csv")
print("Listo:", datos.shape[0], "estudiantes y", datos.shape[1], "columnas. Keras", keras.__version__)

In [ ]:
# @title Función para dibujar las curvas de entrenamiento
def curvas(historia, titulo):
    """Pérdida y exactitud por época, en entrenamiento y en validación."""
    h = historia.history
    epocas = range(1, len(h["loss"]) + 1)
    fig, ejes = plt.subplots(1, 2, figsize=(11, 3.6))
    for eje, metrica, nombre in ((ejes[0], "loss", "Pérdida"), (ejes[1], "accuracy", "Exactitud")):
        eje.plot(epocas, h[metrica], label="Entrenamiento", color="#000000", lw=2)
        eje.plot(epocas, h["val_" + metrica], label="Validación", color="#CB318E", lw=2)
        eje.set_xlabel("Época"); eje.set_title(nombre); eje.grid(alpha=0.3)
        eje.spines[["top", "right"]].set_visible(False)
    mejor = int(np.argmin(h["val_loss"]))
    ejes[0].axvline(mejor + 1, color="#7FC200", ls="--", lw=1.5, label=f"Menor pérdida de validación (época {mejor + 1})")
    ejes[0].legend(frameon=False, fontsize=9)
    fig.suptitle(titulo, fontweight="bold"); fig.tight_layout(); plt.show()
    return mejor + 1

---
## 1. Los datos y la partición

### 1.1 Un primer vistazo

In [ ]:
datos.head()

In [ ]:
conteo = datos["resultado"].value_counts()
print(conteo.to_string(), "\n")
print(f"Si respondemos siempre “{conteo.index[0]}”, acertamos el {conteo.iloc[0] / len(datos):.1%} de las veces.")

Esa es nuestra **línea base**: cualquier modelo que entrenemos tiene que superarla.

### 1.2 Números que no son cantidades

Varias columnas guardan **códigos**. En `ocupacion_madre`, el 5 no es “más” que el 4: cada número
es una ocupación distinta. Si se los pasamos así a la red, va a tratar el código como una cantidad.

In [ ]:
print(sorted(datos["ocupacion_madre"].unique().tolist()))

La solución es la **codificación one-hot**: una columna de ceros y unos por cada valor posible. Hacemos
lo mismo con las columnas de texto, como `programa` o `estado_civil`.

In [ ]:
CATEGORICAS = ["estado_civil", "modo_ingreso", "programa", "titulo_previo", "nacionalidad",
               "titulo_madre", "titulo_padre", "ocupacion_madre", "ocupacion_padre"]
X = pd.get_dummies(datos.drop(columns="resultado"), columns=CATEGORICAS, dtype="float32")
print("Columnas antes:", datos.shape[1] - 1, "· después del one-hot:", X.shape[1])
X.iloc[:3, -8:]

### 1.3 Las etiquetas

La red no recibe palabras: cada clase se vuelve un número, que es la posición de su neurona en la
capa de salida.

In [ ]:
CLASES = ["abandono", "matriculado", "graduado"]
y = datos["resultado"].map({c: i for i, c in enumerate(CLASES)}).to_numpy()
print(datos["resultado"].head(5).tolist(), "→", y[:5])

### 1.4 Tres particiones

- **Entrenamiento** (70 %): la red ajusta sus pesos con estos datos.
- **Validación** (15 %): la usamos para comparar arquitecturas y decidir cuántas épocas entrenar.
- **Prueba** (15 %): la guardamos hasta el final y la usamos **una sola vez**.

`stratify=y` mantiene la proporción de cada clase en las tres partes.

In [ ]:
X_resto, X_prueba, y_resto, y_prueba = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
X_ent, X_val, y_ent, y_val = train_test_split(X_resto, y_resto, test_size=0.15 / 0.85, stratify=y_resto, random_state=42)
print(f"Entrenamiento: {len(X_ent)} · Validación: {len(X_val)} · Prueba: {len(X_prueba)}")

### 1.5 Escalar con las estadísticas del entrenamiento

`edad_ingreso` va de 17 a 70 y `pib` de -4 a 4. Con escalas tan distintas, la red aprende peor.
Restamos la media y dividimos por la desviación estándar, **calculadas solo con el entrenamiento**.
Si usáramos también la validación o la prueba, la red recibiría información de los datos con los que
la vamos a evaluar. Las columnas del one-hot quedan como están (ceros y unos).

In [ ]:
NUMERICAS = [c for c in X.columns if X[c].nunique() > 2]
media, desv = X_ent[NUMERICAS].mean(), X_ent[NUMERICAS].std().replace(0, 1)

def escalar(df, media, desv):
    df = df.copy()
    df[NUMERICAS] = (df[NUMERICAS] - media) / desv
    return df.astype("float32")

X_ent_e, X_val_e, X_prueba_e = (escalar(d, media, desv) for d in (X_ent, X_val, X_prueba))
X_ent_e[NUMERICAS[:5]].describe().loc[["mean", "std"]].round(2)

---
## 2. Un primer MLP

Una red grande: tres capas ocultas de 256 neuronas con ReLU. La capa de salida tiene **una neurona
por clase** y activación **softmax**, que convierte las tres salidas en probabilidades que suman 1.

In [ ]:
keras.utils.set_random_seed(7)
modelo_grande = keras.Sequential([
    keras.Input(shape=(X_ent_e.shape[1],)),
    keras.layers.Dense(256, activation="relu"),
    keras.layers.Dense(256, activation="relu"),
    keras.layers.Dense(256, activation="relu"),
    keras.layers.Dense(len(CLASES), activation="softmax"),
])
modelo_grande.summary()
print(f"{modelo_grande.count_params():,} parámetros para {len(X_ent_e):,} estudiantes de entrenamiento.")

- **Pérdida:** `sparse_categorical_crossentropy` castiga una probabilidad baja en la clase correcta.
- **Optimizador:** Adam ajusta los pesos después de cada lote (`batch_size=32`).
- **Época:** una pasada completa por los datos de entrenamiento.

In [ ]:
modelo_grande.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist_grande = modelo_grande.fit(X_ent_e, y_ent, validation_data=(X_val_e, y_val),
                                epochs=100, batch_size=32, verbose=0)
print("Entrenamiento terminado.")

In [ ]:
mejor_grande = curvas(hist_grande, "MLP grande: 3 capas de 256")
h = hist_grande.history
print(f"Exactitud final · entrenamiento: {h['accuracy'][-1]:.1%} · validación: {h['val_accuracy'][-1]:.1%}")
print(f"Pérdida de validación · mínima: {min(h['val_loss']):.3f} (época {mejor_grande}) · final: {h['val_loss'][-1]:.3f}")

**Para pensar.**

- ¿En qué época es más baja la pérdida de validación? ¿Qué hace después?
- La exactitud de entrenamiento llega a 100 %. ¿Qué aprendió la red?
- ¿Por qué la pérdida de validación sube tanto, si la exactitud de validación casi no cambia?

### 2.1 Lo que sale de la capa softmax

In [ ]:
probs = modelo_grande.predict(X_val_e, verbose=0)
tabla = pd.DataFrame(probs[:5].round(3), columns=CLASES)
tabla["suma"] = tabla.sum(axis=1).round(3)
tabla["real"] = [CLASES[i] for i in y_val[:5]]
display(tabla)
errores = probs.argmax(1) != y_val
print(f"De {errores.sum()} errores en validación, {(probs.max(1)[errores] > 0.95).sum()} tienen más de 95 % de confianza.")

**Para pensar.**

- Cada fila suma 1. ¿Qué tiene que pasar con las otras dos probabilidades si una sube?
- La pérdida castiga con fuerza un error con 99 % de confianza. ¿Qué relación tiene eso con la curva de validación?

---
## 3. Ajustar la red

Tres cambios:

1. **Menos capacidad:** dos capas pequeñas, de 32 y 16 neuronas.
2. **Dropout:** en cada paso de entrenamiento apaga al azar el 30 % de las neuronas, para que la red
   no dependa de unas pocas.
3. **Parada temprana (early stopping):** vigila la pérdida de validación, se detiene cuando lleva 20
   épocas sin mejorar y recupera los pesos de la mejor época.

In [ ]:
def crear_modelo_pequeno(n_entradas):
    modelo = keras.Sequential([
        keras.Input(shape=(n_entradas,)),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(len(CLASES), activation="softmax"),
    ])
    modelo.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return modelo

keras.utils.set_random_seed(7)
modelo_pequeno = crear_modelo_pequeno(X_ent_e.shape[1])
parada = keras.callbacks.EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True)
hist_pequeno = modelo_pequeno.fit(X_ent_e, y_ent, validation_data=(X_val_e, y_val),
                                  epochs=300, batch_size=32, verbose=0, callbacks=[parada])
print(f"{modelo_pequeno.count_params():,} parámetros. Se detuvo en la época {len(hist_pequeno.history['loss'])}.")

In [ ]:
MEJOR_EPOCA = curvas(hist_pequeno, "MLP pequeño con dropout y parada temprana")
comparacion = pd.DataFrame({
    "parámetros": [modelo_grande.count_params(), modelo_pequeno.count_params()],
    "mejor época": [mejor_grande, MEJOR_EPOCA],
    "pérdida de validación mínima": [min(hist_grande.history["val_loss"]), min(hist_pequeno.history["val_loss"])],
    "exactitud de validación en esa época": [hist_grande.history["val_accuracy"][mejor_grande - 1],
                                              hist_pequeno.history["val_accuracy"][MEJOR_EPOCA - 1]],
}, index=["MLP grande", "MLP pequeño"])
comparacion.round(3)

**Para pensar.**

- ¿Cómo se ven ahora las dos curvas de pérdida? ¿Dónde se separan?
- Con la validación elegimos la arquitectura y la época. ¿Podemos usar esa misma exactitud como estimación final? ¿Por qué no?

---
## 4. Entrenamiento final y prueba

Ya decidimos la arquitectura y el número de épocas. Ahora:

1. Juntamos entrenamiento y validación: más datos para el modelo final.
2. Volvemos a calcular la media y la desviación con esos datos juntos.
3. Entrenamos una red nueva con la misma arquitectura durante `MEJOR_EPOCA` épocas (ya no hay
   validación para la parada temprana).
4. Evaluamos **una sola vez** con la prueba.

In [ ]:
X_final = pd.concat([X_ent, X_val])
y_final = np.concatenate([y_ent, y_val])
media_f, desv_f = X_final[NUMERICAS].mean(), X_final[NUMERICAS].std().replace(0, 1)

keras.utils.set_random_seed(7)
modelo_final = crear_modelo_pequeno(X_final.shape[1])
modelo_final.fit(escalar(X_final, media_f, desv_f), y_final, epochs=MEJOR_EPOCA, batch_size=32, verbose=0)

perdida, exactitud = modelo_final.evaluate(escalar(X_prueba, media_f, desv_f), y_prueba, verbose=0)
print(f"Prueba · exactitud: {exactitud:.1%} (línea base: {np.mean(y_prueba == 2):.1%}) · pérdida: {perdida:.3f}")

### 4.1 ¿En qué clase se equivoca?

In [ ]:
pred = modelo_final.predict(escalar(X_prueba, media_f, desv_f), verbose=0).argmax(1)
matriz = pd.crosstab(pd.Series([CLASES[i] for i in y_prueba], name="real"),
                     pd.Series([CLASES[i] for i in pred], name="predicho")).reindex(index=CLASES, columns=CLASES)
display(matriz)
for i, c in enumerate(CLASES):
    print(f"Sensibilidad (recall) de “{c}”: {np.mean(pred[y_prueba == i] == i):.1%}")

**Para pensar.**

- ¿Qué clase encuentra peor la red? ¿Por qué crees que le cuesta?
- Si la institución quiere ofrecer apoyo a quien va a abandonar, ¿qué error le cuesta más?

---
## Cierre

- Partimos los datos en tres y escalamos con las estadísticas del entrenamiento.
- La red grande memorizó: 100 % en entrenamiento y una pérdida de validación que sube desde el principio.
- Con menos capacidad, dropout y parada temprana, las curvas de validación mejoraron.
- Con la validación elegimos; con la prueba medimos, una sola vez.

**Fuente de los datos:** Realinho, V., Vieira Martins, M., Machado, J. y Baptista, L. (2021).
*Predict Students' Dropout and Academic Success* [conjunto de datos]. UCI Machine Learning Repository.
https://doi.org/10.24432/C5MC89. Licencia CC BY 4.0. Renombramos las columnas al español y pusimos
nombre al estado civil y al programa.